# The connection between a 2-mode coupler and a 2-mode multimode system

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. 

In [1]:
from sympy import *
from time import time
from itertools import combinations_with_replacement

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi, varsigma) = symbols(
    '''delta, nu, Aeff, chi, varsigma'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

## Elliptic function identities

In [2]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*zeta(m*omega[3] + n*omega[1], g2, g3))
)

# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio
quasi_period_sigma_b

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(\wp'(x, g2*c**4, g3*c**6), \wp'(x*c, g2, g3)*c**3)

Eq(omega[1], f(1, g2, g3))

Eq(omega[1]*c, f(1, g2/c**4, g3/c**6))

Eq(omega[3], f(3, g2, g3))

Eq(omega[3]*c, f(3, g2/c**4, g3/c**6))

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [3]:
uhat_muj = Eq(
    uhat(z, mu[j]),
    sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))
    *sqrt(wpp(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)
    *exp(-(-1)**j*z*beta[0] + z*r[0, j] + epsilon[j])/
    (
        sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
        *sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))
        *sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**Rational(1/4)
    )
)
vhat_muj = Eq(
    vhat(z, mu[j]),
    sqrt(wpp(z1, g2, g3))*sigma(z - mu[j], g2, g3)
    *exp((-1)**j*z*beta[0] - z*r[0, j] - epsilon[j])/
    (
        sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))
        *sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
        *sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))
        *sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**Rational(1/4)
    )
)
uvj_hat_wp = Eq(
    uhat_muj.lhs*vhat_muj.lhs, 
    (uhat_muj.rhs*vhat_muj.rhs).simplify()
    .subs([
        (
            sigma_p_identity.rhs.subs([(x,z-z0), (y, mu[j] - z0)]),
            sigma_p_identity.lhs.subs([(x,z-z0), (y, mu[j] - z0)])
        )
    ])
)

uhat_muj
vhat_muj
uvj_hat_wp

Eq(uhat(z, mu[j]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(-(-1)**j*z*beta[0] + z*r[0, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp((-1)**j*z*beta[0] - z*r[0, j] - epsilon[j])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), (-wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))*\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sqrt(d[4])))

In [4]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]


sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

sum_r1j_1
r1j_log_part

for _ in Lams:
    _

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

### The original parameters

In [108]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

sqrt_d4_b2 = Eq(sqrt(d[4]), b[2])


g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _
    
sqrt_d4_b2

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(sqrt(d[4]), b[2])

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

In [6]:
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(gamma[j] - lamda[1])*sqrt(d[4])
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1])
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)*wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2,
    (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4])
)


wp_z1_lam_uvhat_j = Eq(
    wpp(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])),
    -lamda[1] - Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2
)
wp_z1_lam_uvhat_j_sqrt = Eq(
    sqrt(wpp(z1, g2, g3))/sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))/sqrt(sqrt(d[4])),
    I*varsigma*sqrt(lamda[1] + Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)
)
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)
wppz1_djs = Eq( 4*wpp(z1, g2, g3)/sqrt(d[4]), -d[5])

wp_z0_mu_j_d = Eq(
    wp(-z0 + mu[j], g2, g3),
    d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - 
    (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1]))
)
wpp_z0_mu_j_d = Eq(
    wpp(-z0 + mu[j], g2, g3),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)
    *(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2)
)
wp_z1_z0_mu_j_d5_lam = Eq(
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.lhs)*wppz1_djs.lhs*sqrt(d[4])/4, 
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.rhs)*wppz1_djs.rhs*sqrt(d[4])/4
)

wppz1_djs_sqrt = Eq(
    sqrt(fraction(wppz1_djs.lhs)[0])/sqrt(fraction(wppz1_djs.lhs)[1]), 
    I*varsigma*sqrt(-wppz1_djs.rhs)
)



B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
wp_z1_z0_mu_j_d5_lam
wp_z1_lam_uvhat_j
wp_z1_lam_uvhat_j_sqrt
wppz1_djs
wppz1_djs_sqrt
wp_z0_mu_j_d
wpp_z0_mu_j_d

d4_lam_0
d5_dj


Eq(\wp'(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-gamma[j] + lamda[1])*sqrt(d[4]))

Eq(\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)/(gamma[j] - lamda[1]))

Eq(\wp'(z1, g2, g3)*\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2, (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4]))

Eq(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3), d[5]/(4*(-gamma[j] + lamda[1])))

Eq(\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])), -lamda[1] - Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)

...

Eq(2*sqrt(\wp'(z1, g2, g3))/d[4]**(1/4), I*varsigma*sqrt(d[5]))

Eq(wp(-z0 + mu[j], g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*gamma[j] - 4*lamda[1]))

Eq(\wp'(-z0 + mu[j], g2, g3), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)*(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2))

Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)

Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [7]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

gamma_lamda_bj = Eq(
    drhop_b.rhs.subs(rho(z), lamda[1]).doit()
    .subs(gamma[2], - gamma[1])
    .expand().collect(4*gamma[1]**2 - 4*lamda[1]**2, factor),
    0
)
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, solve(gamma_lamda_bj, gamma[1]**2)[0])

drhop_b
drhop_d
drho_2z
drho_2z_b
gamma_lamda_bj_gam1

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j - 1)*d[j]/2, (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) - 4*rho(z) + 2*gamma[1] + 2*gamma[2])

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

In [8]:
bs_as_ds = [_.subs([_.args for _ in c_j_coeffs]) for _ in d_j_coeffs]

b0_d1 = Eq(b[0], solve(bs_as_ds[1], b[0])[0])
b2_d3 = Eq(b[2], solve(bs_as_ds[3], b[2])[0])
b1_sqrd_as_d = Eq(
    b[1]**2,
    solve(
        bs_as_ds[2]
        .subs([
            b0_d1.args, 
            (2*bs_as_ds[4].rhs/bs_as_ds[3].rhs, 2*bs_as_ds[4].lhs/bs_as_ds[3].lhs)
        ]),
        b[1]**2
    )[0]
)
b_to_d_subs = [
    b0_d1.args,
    b2_d3.args,
    b1_sqrd_as_d.args
]

for _ in bs_as_ds:
    _
print()
for _ in b_to_d_subs:
    Eq(*_)
    
def generate_lambda_reductions(max_power=12):
    """
    Generate substitutions reducing lamda**n (n >= 4)
    to expressions at most cubic in lamda.
    """
    subs = {}

    # base reduction: λ⁴
    subs[lamda[1]**4] = -(d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3) / d[4]

    # build higher powers
    for _n in range(5, max_power + 1):
        subs[lamda[1]**_n] = expand(lamda * subs[lamda[1]**(_n - 1)]).subs(subs)

    return subs

lamda_d_reductions = generate_lambda_reductions()

Eq(d[0], b[0]**2 - 4*Product(gamma[j], (j, 1, 2)))

Eq(d[1], 2*b[0]*b[1])

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4)

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(b[0], d[1]/(2*b[1]))

Eq(b[2], d[3]/(2*b[1]))

Eq(b[1]**2, -2*d[1]*d[4]/d[3] + d[2] + 4)

### The original equations of motion and conserved quantities

In [9]:
duhatj = Eq(
    Derivative(uhat(z, mu[j]), z),
    -uhat(z, mu[j])**2*vhat(z, mu[j])*b[2] + uhat(z, mu[j])*b[1]/2 
    + Product(vhat(z, mu[k]), (k,1,2))/vhat(z, mu[j])
)
dvhatj = Eq(
    Derivative(vhat(z, mu[j]), z),
    vhat(z, mu[j])**2*uhat(z, mu[j])*b[2] - vhat(z, mu[j])*b[1]/2 
    - Product(uhat(z, mu[k]), (k,1,2))/uhat(z, mu[j])
)

du_dv_hats = [
    duhatj.subs(j,1).doit(),
    duhatj.subs(j,2).doit(),
    dvhatj.subs(j,1).doit(),
    dvhatj.subs(j,2).doit(),
]
du_dv_hat_subs = [_.args for _ in du_dv_hats]


# Hamiltonian

Hhat_uv = Eq(
    Hhat, 
    Sum(-b[2]/2*(uhat(z, mu[j])*vhat(z, mu[j]))**2 + b[1]/2*uhat(z, mu[j])*vhat(z, mu[j]), (j,1,2)) +
    uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2])
)
Hhat_b0 = Eq(Hhat, -(gamma[1] - gamma[2])**2*b[2]/4 + b[0])
assert diff(Hhat_uv.rhs.doit(),z).subs(du_dv_hat_subs).simplify() == 0, 'Hamiltonian not conserved'
 
# Intermodal power conservation (gamma_j and rho were unchanged when transforming to hat)

uvhat_j_rhohat = Eq(uhat(z,mu[j])*vhat(z,mu[j]), gamma[j] - rhohat(z))
uvhat_j_interm = Eq(
    uvhat_j_rhohat.lhs.subs(j,1) - uvhat_j_rhohat.lhs.subs(j,2),
    uvhat_j_rhohat.rhs.subs(j,1) - uvhat_j_rhohat.rhs.subs(j,2)
)
assert diff(uvhat_j_interm.lhs.doit(),z).subs(du_dv_hat_subs).simplify() == 0, 'Intermodal power not conserved'

duhatj
dvhatj
Hhat_uv
Hhat_b0
uvhat_j_rhohat
uvhat_j_interm

Eq(Derivative(uhat(z, mu[j]), z), -uhat(z, mu[j])**2*vhat(z, mu[j])*b[2] + uhat(z, mu[j])*b[1]/2 + Product(vhat(z, mu[k]), (k, 1, 2))/vhat(z, mu[j]))

Eq(Derivative(vhat(z, mu[j]), z), uhat(z, mu[j])*vhat(z, mu[j])**2*b[2] - vhat(z, mu[j])*b[1]/2 - Product(uhat(z, mu[k]), (k, 1, 2))/uhat(z, mu[j]))

Eq(Hhat, uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2)))

Eq(Hhat, -(gamma[1] - gamma[2])**2*b[2]/4 + b[0])

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -rhohat(z) + gamma[j])

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

## The transform to 2-mode multimode like dynamics

We will now explore a nonlinear transformation that removes the square root of the $z$ dependent $\wp$ term from the denominator of the solutions. We will not yet do anything to the phase part so we anticipate solutions to be the product of a Kronecker theta function and an exponential factor with an essential singularities. There will be two modes not three, unlike the transform to degenerate FWM considered previously in relation to the coherent coupler.

In [10]:
uhat_muj
vhat_muj
uvj_hat_wp

Eq(uhat(z, mu[j]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(-(-1)**j*z*beta[0] + z*r[0, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp((-1)**j*z*beta[0] - z*r[0, j] - epsilon[j])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), (-wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))*\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sqrt(d[4])))

We thus define new modes in bar coordinates as follows:

In [11]:
ubar_sigma_j = Eq(
    ubar(z, mu[j]), 
    sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(
        -(-1)**j*z*beta[0] + z*r[0, j] + 
        log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2 
        + epsilon[j]
    )
)
vbar_sigma_j = Eq(
    vbar(z, mu[j]), 
    sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(
        (-1)**j*z*beta[0] - z*r[0, j] + 
        - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2 
        - epsilon[j]
    )
)
uvbar_wp_j = Eq(
    ubar_sigma_j.lhs*vbar_sigma_j.lhs, 
    (ubar_sigma_j.rhs*vbar_sigma_j.rhs)
    .subs([
        (
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).rhs,
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).lhs
        )
    ])
    .simplify()
)
uvbar_j_d5_gam_wp_z1 = Eq(
    uvbar_wp_j.lhs + wp_z1_z0_mu_j_d5_lam.rhs,
    uvbar_wp_j.rhs + wp_z1_z0_mu_j_d5_lam.lhs
)
uvhat_j_to_uvbar_j = uvj_hat_wp.subs([
    (uvbar_wp_j.rhs, uvbar_wp_j.lhs),
    (uvbar_j_d5_gam_wp_z1.rhs, uvbar_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args
])
uvbar_12_d5_gam_wp_z1 = Eq(
    (uvbar_j_d5_gam_wp_z1.lhs*(gamma[j] - lamda[1])).expand().collect(d[5], factor),
    uvbar_j_d5_gam_wp_z1.rhs*(gamma[j] - lamda[1])
)
uvbar_12_d5_gam_wp_z1 = Eq(
    uvbar_12_d5_gam_wp_z1.lhs.subs(j,1) + uvbar_12_d5_gam_wp_z1.lhs.subs(j,2),
    (uvbar_12_d5_gam_wp_z1.rhs.subs(j,1) + uvbar_12_d5_gam_wp_z1.rhs.subs(j,2))
    .expand().subs(gamma[2], - gamma[1]).factor()
)

uvbar12_to_uvbarj = uvbar_12_d5_gam_wp_z1.subs(uvbar_j_d5_gam_wp_z1.rhs, uvbar_j_d5_gam_wp_z1.lhs)
uvbar12_to_uvbarj = Eq(
    -uvbar12_to_uvbarj.rhs/(2*lamda[1]),
    -uvbar12_to_uvbarj.lhs/(2*lamda[1])
)


ubar_sigma_j
vbar_sigma_j
uvbar_wp_j
uvbar_j_d5_gam_wp_z1
uvbar_12_d5_gam_wp_z1
uvbar12_to_uvbarj
uvhat_j_to_uvbar_j

Eq(ubar(z, mu[j]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(-(-1)**j*z*beta[0] + z*r[0, j] + epsilon[j])/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)))

Eq(vbar(z, mu[j]), sigma(z - mu[j], g2, g3)*exp((-1)**j*z*beta[0] - z*r[0, j] - epsilon[j])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)))

Eq(ubar(z, mu[j])*vbar(z, mu[j]), -wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))

Eq(ubar(z, mu[j])*vbar(z, mu[j]) + d[5]/(4*(-gamma[j] + lamda[1])), wp(z1, g2, g3) - wp(z - z0, g2, g3))

Eq((gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) - d[5]/2, 2*(-wp(z1, g2, g3) + wp(z - z0, g2, g3))*lamda[1])

Eq(ubar(z, mu[j])*vbar(z, mu[j]) + d[5]/(4*(-gamma[j] + lamda[1])), (-(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2)/(2*lamda[1]))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -(-gamma[j] + lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])/(ubar(z, mu[j])*vbar(z, mu[j]) + d[5]/(-4*gamma[j] + 4*lamda[1])))

The following modal power relations exist between coordinates and we note a Mobius transform exists between any modal power in one coordinate system and any modal power in the other. We introduce the auxiliary function $h(z)$.

In [12]:
# Relating sums of mode powers in both coordinates
uvbar12_sum_to_uvhat12_sum = Eq(
    (1/wp_z1_lam_uvhat_j.lhs)
    .subs([
        (uvbar_12_d5_gam_wp_z1.rhs/(2*lamda[1]), uvbar_12_d5_gam_wp_z1.lhs/(2*lamda[1]))
    ])*wppz1_djs.lhs,
    wppz1_djs.rhs/wp_z1_lam_uvhat_j.rhs
)

uvhat_j_to_uvbar_j_sym = uvhat_j_to_uvbar_j.subs([(uvbar12_to_uvbarj.lhs.expand(), uvbar12_to_uvbarj.rhs)])

uvhat_to_uvbar_sum = Eq(
    (2*d[5]/uvbar12_sum_to_uvhat12_sum.rhs).doit(), 
    2*d[5]/uvbar12_sum_to_uvhat12_sum.lhs
)

h_u_v_bar = Eq(
    h(z),
    -(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])
    - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2
)

h_as_u_v = Eq(
    uvbar12_sum_to_uvhat12_sum.lhs.subs(d[5], solve(h_u_v_bar, d[5])[0]).simplify()*lamda[1]/2,
    uvbar12_sum_to_uvhat12_sum.rhs*lamda[1]/2
)

uvbar_j_rho = Eq(ubar(z, mu[j])*vbar(z, mu[j]), gammabar[j] - rhobar(z))
uv_bars_to_rhobar_subs = [
    uvbar_j_rho.subs(j,1).args, 
    uvbar_j_rho.subs(j,2).args
]

gams2_to_1_subs = [
    (gamma[2], -gamma[1]),
    (gammabar[2], - gammabar[1])
]

h_as_rhobar = Eq(
    h_u_v_bar.lhs, 
    h_u_v_bar.rhs
    .subs(uv_bars_to_rhobar_subs)
    .subs(gams2_to_1_subs)
    .expand().collect(rhobar(z), factor)
)

rhobar_as_h = Eq(rhobar(z), solve(h_as_rhobar, rhobar(z))[0])

_d5_const = d[5]/4/(gamma[j] - lamda[1])
uvbarj_as_h = Eq(
    (uvbar12_to_uvbarj.lhs + _d5_const).simplify(),
    (
        uvbar12_to_uvbarj.rhs
        - (h_u_v_bar.rhs - h_u_v_bar.lhs)/(2*lamda[1])
        + _d5_const
    )
    .expand().collect(d[5], factor)
    
)



uvbar12_sum_to_uvhat12_sum
uvhat_to_uvbar_sum
uvhat_j_to_uvbar_j_sym
h_u_v_bar
h_as_u_v
uvbar_j_rho
h_as_rhobar
rhobar_as_h
uvbarj_as_h

Eq(-2*((gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) - d[5]/2)/lamda[1], -d[5]/(-lamda[1] - Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2))

Eq(uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]) + 2*lamda[1], -d[5]*lamda[1]/((gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) - d[5]/2))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -2*(-gamma[j] + lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*lamda[1]/(-(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2))

Eq(h(z), (-gamma[1] + lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2)

Eq(h(z), -d[5]*lamda[1]/(2*(-lamda[1] - Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)))

Eq(ubar(z, mu[j])*vbar(z, mu[j]), -rhobar(z) + gammabar[j])

Eq(h(z), (d[5] - 4*gamma[1]*gammabar[1])/2 - 2*rhobar(z)*lamda[1])

Eq(rhobar(z), (-h(z)/2 + d[5]/4 - gamma[1]*gammabar[1])/lamda[1])

Eq(ubar(z, mu[j])*vbar(z, mu[j]), h(z)/(2*lamda[1]) + d[5]/(4*(gamma[j] - lamda[1])))

The modes in each coordinate system are thus related to each other as follows:

In [13]:
# Mode transforms
_sqrt_d5_id = Eq(
    sqrt(d[5])/sqrt(d[5]/(-gamma[j]+lamda[1])),
    I*sqrt(gamma[j] - lamda[1])
)

to_bar_transform_subs =[
    (uvbar_wp_j.rhs, uvbar_wp_j.lhs),
    (uvbar_j_d5_gam_wp_z1.rhs, uvbar_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args,
    (ubar_sigma_j.rhs, ubar_sigma_j.lhs),
    (vbar_sigma_j.rhs, vbar_sigma_j.lhs),
    wppz1_djs_sqrt.args,
#     uvbar_integral_phiz.args,
    r1j_log_part.args,
    _sqrt_d5_id.args
]
uj_as_ubarj = uhat_muj.subs(to_bar_transform_subs)
vj_as_vbarj = vhat_muj.subs(to_bar_transform_subs)

u_v_j_as_bars_subs = [
    uj_as_ubarj.subs(j,1).args,
    uj_as_ubarj.subs(j,2).args,
    vj_as_vbarj.subs(j,1).args,
    vj_as_vbarj.subs(j,2).args
]

uvbar12_to_uvbarj_sqrt = Eq(
    2*sqrt(uvbar12_to_uvbarj.lhs).simplify(), 
    2*sqrt(fraction(uvbar12_to_uvbarj.rhs)[0])/sqrt(fraction(uvbar12_to_uvbarj.rhs)[1])
)
uj_as_ubarj_b = Eq(
    uj_as_ubarj.lhs, 
    uj_as_ubarj.rhs
    .simplify()
    .subs([uvbar12_to_uvbarj_sqrt.args])
)
vj_as_vbarj_b = Eq(
    vj_as_vbarj.lhs, 
    vj_as_vbarj.rhs
    .simplify()
    .subs([uvbar12_to_uvbarj_sqrt.args])
)

# Alternative version
_sqrt_d5_id_b = Eq(
    sqrt(d[5]/(-gamma[j]+lamda[1])),
    I*sqrt(d[5]/(gamma[j]-lamda[1]))
)

to_bar_transform_subs_b =[
    wp_z1_z0_mu_j_d5_lam.args,
    (ubar_sigma_j.rhs, ubar_sigma_j.lhs),
    (vbar_sigma_j.rhs, vbar_sigma_j.lhs),
#     uvbar_integral_phiz.args,
    r1j_log_part.args,
    _sqrt_d5_id_b.args
]
ubarj_as_uj_b = uhat_muj.subs(to_bar_transform_subs_b)
vbarj_as_vj_b = vhat_muj.subs(to_bar_transform_subs_b)

ubarj_as_uj_b = Eq(
    ubar(z, mu[j]), 
    varsigma**2*solve(ubarj_as_uj_b, ubar(z, mu[j]))[0]
    .subs([(1/wp_z1_lam_uvhat_j_sqrt.lhs, 1/wp_z1_lam_uvhat_j_sqrt.rhs)])
)
vbarj_as_vj_b = Eq(
    vbar(z, mu[j]), 
    varsigma**2*solve(vbarj_as_vj_b, vbar(z, mu[j]))[0]
    .subs([(1/wp_z1_lam_uvhat_j_sqrt.lhs, 1/wp_z1_lam_uvhat_j_sqrt.rhs)])
)

u_v_j_as_bars_subs_b = [
    uj_as_ubarj_b.subs(j,1).doit().args,
    uj_as_ubarj_b.subs(j,2).doit().args,
    vj_as_vbarj_b.subs(j,1).doit().args,
    vj_as_vbarj_b.subs(j,2).doit().args
]

uj_as_ubarj_h = uj_as_ubarj_b.expand().subs([
    (
        h_u_v_bar.rhs.expand(), 
        h_u_v_bar.lhs
    )
])
vj_as_vbarj_h = vj_as_vbarj_b.expand().subs([
    (
        h_u_v_bar.rhs.expand(), 
        h_u_v_bar.lhs
    )
])

u_v_j_as_bars_h_subs = [
    uj_as_ubarj_h.subs(j,1).doit().args,
    uj_as_ubarj_h.subs(j,2).doit().args,
    vj_as_vbarj_h.subs(j,1).doit().args,
    vj_as_vbarj_h.subs(j,2).doit().args
]

uvhat_to_uvbar_j = Eq(
    uvhat_j_to_uvbar_j.lhs, 
    uvhat_j_to_uvbar_j.lhs
    .subs([uj_as_ubarj_h.args, vj_as_vbarj_h.args, (varsigma**2, 1)])
)

# h as uvbar but no const
h_uvbar12_no_const = Eq(
    ((uvhat_to_uvbar_j.lhs.subs(j,1) - uvhat_to_uvbar_j.lhs.subs(j,2))*h(z))
    .subs([uvhat_j_interm.args])/uvhat_j_interm.rhs, 
    ((uvhat_to_uvbar_j.rhs.subs(j,1) - uvhat_to_uvbar_j.rhs.subs(j,2))*h(z))
    .collect(h(z))/uvhat_j_interm.rhs
)


uj_as_ubarj
vj_as_vbarj
print()
uj_as_ubarj_b
vj_as_vbarj_b
print()
uj_as_ubarj_h
vj_as_vbarj_h
print()
ubarj_as_uj_b
vbarj_as_vj_b
print()
uvhat_to_uvbar_j
print()
h_uvbar12_no_const

Eq(uhat(z, mu[j]), -varsigma*sqrt(gamma[j] - lamda[1])*ubar(z, mu[j])/sqrt(ubar(z, mu[j])*vbar(z, mu[j]) + d[5]/(-4*gamma[j] + 4*lamda[1])))

Eq(vhat(z, mu[j]), -varsigma*sqrt(gamma[j] - lamda[1])*vbar(z, mu[j])/sqrt(ubar(z, mu[j])*vbar(z, mu[j]) + d[5]/(-4*gamma[j] + 4*lamda[1])))

Eq(uhat(z, mu[j]), -sqrt(2)*varsigma*sqrt(gamma[j] - lamda[1])*ubar(z, mu[j])*sqrt(lamda[1])/sqrt(-(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2))

Eq(vhat(z, mu[j]), -sqrt(2)*varsigma*sqrt(gamma[j] - lamda[1])*vbar(z, mu[j])*sqrt(lamda[1])/sqrt(-(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2))

Eq(uhat(z, mu[j]), -sqrt(2)*varsigma*sqrt(gamma[j] - lamda[1])*ubar(z, mu[j])*sqrt(lamda[1])/sqrt(h(z)))

Eq(vhat(z, mu[j]), -sqrt(2)*varsigma*sqrt(gamma[j] - lamda[1])*vbar(z, mu[j])*sqrt(lamda[1])/sqrt(h(z)))

Eq(ubar(z, mu[j]), varsigma*sqrt(d[5]/(gamma[j] - lamda[1]))*uhat(z, mu[j])/(2*sqrt(lamda[1] + Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)))

Eq(vbar(z, mu[j]), varsigma*sqrt(d[5]/(gamma[j] - lamda[1]))*vhat(z, mu[j])/(2*sqrt(lamda[1] + Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), 2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*lamda[1]/h(z))

Eq(h(z), (2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])/(gamma[1] - gamma[2]))

The modes in the bar coordinate system inherit intermodal power conservation laws as below:

In [14]:
# Intermodal power conservation laws in both coordinates

uvhat_to_rhohat_subs = [
    uvhat_j_rhohat.subs(j,1).args,
    uvhat_j_rhohat.subs(j,2).args
]

uvbar_12_int_const = Eq(
    (uvhat_j_to_uvbar_j.lhs.subs(j,1) - uvhat_j_to_uvbar_j.lhs.subs(j,2))
    .subs(uvhat_to_rhohat_subs),
    uvhat_j_to_uvbar_j.rhs.subs(j,1) - uvhat_j_to_uvbar_j.rhs.subs(j,2)
)
uvbar_12_int_const_b = Eq(
    (uvbar_12_int_const.lhs - uvbar_12_int_const.rhs).factor(),
    0
)
uvbar_12_int_const_b = Eq(
    (uvbar_12_int_const_b.lhs*fraction(uvbar_12_int_const_b.lhs)[1]
     /(d[5]*(gamma[1] - lamda[1])*(gamma[2] - lamda[1]))/4)
    .expand().collect([d[5]], factor),
    0
)

_d5_gam_eq = (d[5]/(gamma[1] - lamda[1]) - d[5]/(gamma[2] - lamda[1]))
uvbar_12_int_const_b = Eq(
    - uvbar_12_int_const_b.rhs + uvbar_12_int_const_b.lhs.subs(d[5],0),
    (- uvbar_12_int_const_b.lhs + uvbar_12_int_const_b.lhs.subs(d[5],0))
    .subs(_d5_gam_eq.factor(), _d5_gam_eq)
)

gammabar12_gamma12 = uvbar_12_int_const_b.subs(uv_bars_to_rhobar_subs)
gammabar_1_to_gamma_1 = Eq(
    gammabar12_gamma12.lhs
    .subs(gams2_to_1_subs)
    .simplify()/2,
    gammabar12_gamma12.rhs
    .subs(gams2_to_1_subs)
    .simplify()/2
)

gammbar_sum_0 = Eq(gammabar[1] + gammabar[2], 0)

rhobar_to_uv12_sum = uvbar12_sum_to_uvhat12_sum.subs(uv_bars_to_rhobar_subs)
rhobar_to_uv12_sum = Eq(
    rhobar_to_uv12_sum.lhs
    .subs(gams2_to_1_subs)
    .expand().collect(rhobar(z), factor)/4, 
    rhobar_to_uv12_sum.rhs/4
)


uvbar_12_int_const
uvbar_12_int_const_b
uvbar_j_rho
gammabar12_gamma12
gammabar_1_to_gamma_1
gammbar_sum_0
rhobar_to_uv12_sum

Eq(gamma[1] - gamma[2], (-gamma[2] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/(-4*gamma[2] + 4*lamda[1])) - (-gamma[1] + lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[1])*vbar(z, mu[1]) + d[5]/(-4*gamma[1] + 4*lamda[1])))

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), -d[5]/(4*(gamma[2] - lamda[1])) + d[5]/(4*(gamma[1] - lamda[1])))

Eq(ubar(z, mu[j])*vbar(z, mu[j]), -rhobar(z) + gammabar[j])

Eq(gammabar[1] - gammabar[2], -d[5]/(4*gamma[2] - 4*lamda[1]) + d[5]/(4*gamma[1] - 4*lamda[1]))

Eq(gammabar[1], d[5]*gamma[1]/(4*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(gammabar[1] + gammabar[2], 0)

Eq((d[5] - 4*gamma[1]*gammabar[1])/(4*lamda[1]) - rhobar(z), -d[5]/(4*(-lamda[1] - Sum(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, 2))/2)))

### The derivative of $h(z)$

In [15]:
_h_factor_subs = Eq((h_u_v_bar.rhs*2).expand(), h_u_v_bar.rhs*2)


assert (
    Derivative(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), z)
    .doit()
    .subs(du_dv_hat_subs)
    .simplify()
) == 0, 'intermodal power not conserved'


dh_as_u_v = Eq(
    diff(h_as_u_v.lhs,z), 
    diff(h_as_u_v.rhs.doit(), z)
    .subs(du_dv_hat_subs)
    .factor()
)
dh_as_u_v_bar = Eq(
    dh_as_u_v.lhs,
    dh_as_u_v.rhs
    .subs([uvhat_to_uvbar_sum.args])
    .subs(u_v_j_as_bars_subs_b)
    .subs(varsigma**2, 1)
    .factor()
    .subs([_h_factor_subs.args])
)
dlog_h_as_u_v_bar = Eq(
    dh_as_u_v_bar.lhs/h_u_v_bar.lhs,
    (dh_as_u_v_bar.rhs/h_u_v_bar.rhs).factor()
)

drhobar = Eq(
    dh_as_u_v_bar.lhs.subs([h_as_rhobar.args]).doit()/(-2*lamda[1]),
    dh_as_u_v_bar.rhs/(-2*lamda[1])
)

for _ in du_dv_hats:
    _
print()

h_as_rhobar
dh_as_u_v
dh_as_u_v_bar
dlog_h_as_u_v_bar
drhobar

Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + uhat(z, mu[1])*b[1]/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + uhat(z, mu[2])*b[1]/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]) - vhat(z, mu[1])*b[1]/2)

Eq(Derivative(vhat(z, mu[2]), z), -uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2] - vhat(z, mu[2])*b[1]/2)

Eq(h(z), (d[5] - 4*gamma[1]*gammabar[1])/2 - 2*rhobar(z)*lamda[1])

Eq(Derivative(h(z), z), 2*(uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]))*d[5]*lamda[1]/(uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]) + 2*lamda[1])**2)

Eq(Derivative(h(z), z), 2*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(2*(-gamma[1] + lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5])/d[5])

Eq(Derivative(h(z), z)/h(z), 4*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(Derivative(rhobar(z), z), -(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(2*(-gamma[1] + lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5])/(d[5]*lamda[1]))

### Transforming the Hamiltonian to hat coordinates

We now transform the original Hamiltonian to the new coordinates. We do not anticipate this will yield the canonical Hamiltonian for the new system of differential equations but it will be closely related:

In [16]:
u_v_hats_as_bars_h_subs = [
    uj_as_ubarj_h.subs(j,1).args,
    uj_as_ubarj_h.subs(j,2).args,
    vj_as_vbarj_h.subs(j,1).args,
    vj_as_vbarj_h.subs(j,2).args
]

a_0_eq_bar = Eq(
    Hhat_uv.lhs
    .subs([Hhat_b0.args]),
    Hhat_uv.rhs
    .replace(*uj_as_ubarj_h.args)
    .replace(*vj_as_vbarj_h.args)
    .subs(u_v_hats_as_bars_h_subs)
    .subs(varsigma**2, 1)
    .doit()
)

a_0_eq_bar = Eq(
    a_0_eq_bar.lhs*h(z)**2,
    sum(_*h(z)**2 for _ in a_0_eq_bar.rhs.args)
)

a_0_eq_bar_b = Eq(
    - (a_0_eq_bar.rhs - a_0_eq_bar.lhs).subs(sqrt(gamma[1]-lamda[1])*sqrt(gamma[2]-lamda[1]), 0),
    (
        (a_0_eq_bar.rhs - a_0_eq_bar.lhs) 
        - (a_0_eq_bar.rhs - a_0_eq_bar.lhs).subs(sqrt(gamma[1]-lamda[1])*sqrt(gamma[2]-lamda[1]), 0)
    ).factor()
)



a_0_eq_bar
a_0_eq_bar_b

Eq((-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*h(z)**2, ((gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]))*h(z)*b[1]*lamda[1] - 2*((gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + (gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2)*b[2]*lamda[1]**2 + 2*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*h(z)*ubar(z, mu[1])*ubar(z, mu[2])*lamda[1] + 2*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*h(z)*vbar(z, mu[1])*vbar(z, mu[2])*lamda[1])

Eq((-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*h(z)**2 - ((gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]))*h(z)*b[1]*lamda[1] + 2*((gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + (gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2)*b[2]*lamda[1]**2, 2*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*h(z)*lamda[1])

At this point we recap that we now have three different expressions for $h(z)$ in terms of modal powers in bar coordinates.

In [17]:
h_as_uvbarj = Eq(h(z), solve(uvbarj_as_h, h(z))[0].expand().collect(d[5], factor))
h_as_uvbarj_sqrd = Eq(
    h_as_uvbarj.lhs**2, 
    sum(
        _.factor() for _ in 
        ((h_as_uvbarj.rhs.subs(j,1)**2 + h_as_uvbarj.rhs.subs(j,2)**2)/2)
        .expand().args
    )
)

h_u_v_bar
h_uvbar12_no_const
h_as_uvbarj
h_as_uvbarj_sqrd

Eq(h(z), (-gamma[1] + lamda[1])*ubar(z, mu[1])*vbar(z, mu[1]) - (gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2]) + d[5]/2)

Eq(h(z), (2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])/(gamma[1] - gamma[2]))

Eq(h(z), 2*ubar(z, mu[j])*vbar(z, mu[j])*lamda[1] - d[5]*lamda[1]/(2*(gamma[j] - lamda[1])))

Eq(h(z)**2, 2*ubar(z, mu[1])**2*vbar(z, mu[1])**2*lamda[1]**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2*lamda[1]**2 - ubar(z, mu[2])*vbar(z, mu[2])*d[5]*lamda[1]**2/(gamma[2] - lamda[1]) + d[5]**2*lamda[1]**2/(8*(gamma[2] - lamda[1])**2) - ubar(z, mu[1])*vbar(z, mu[1])*d[5]*lamda[1]**2/(gamma[1] - lamda[1]) + d[5]**2*lamda[1]**2/(8*(gamma[1] - lamda[1])**2))

We use $\bar{\rho}(z)$ as an intermediary towards a balanced version of the transformed Hamiltonian. We scale everything by some yet undetermined constant $c_0$ which we will use to match with the modal power evolution of our target transformed system:

In [18]:
a_0_eq_bar_rho = Eq(
    a_0_eq_bar_b.lhs
    .subs([h_as_rhobar.args])
    .subs(uv_bars_to_rhobar_subs)
    .expand().collect(rhobar(z), factor)
    .subs(gams2_to_1_subs)
    .subs([gammabar_1_to_gamma_1.args])
    .expand().collect(rhobar(z), factor),
    a_0_eq_bar_b.rhs
)

# rhobar ids
rhobar_as_uvj = Eq(rhobar(z), solve(uvbar_j_rho, rhobar(z))[0])
rhobar_uvj_balanced = Eq(
    -((uvbar_j_rho.rhs.subs(j,1) + uvbar_j_rho.rhs.subs(j,2))/2)
    .subs(gams2_to_1_subs),
    -((uvbar_j_rho.lhs.subs(j,1) + uvbar_j_rho.lhs.subs(j,2))/2)
)
rhobar_sqrd_as_uvj = Eq(
    rhobar_as_uvj.lhs**2, 
    ((rhobar_as_uvj.rhs.subs(j,1)**2 + rhobar_as_uvj.rhs.subs(j,2)**2).expand()/2)
    .subs(gams2_to_1_subs)
    .subs([gammabar_1_to_gamma_1.args])
    .collect(d[5], factor)
    .subs([uvbar_12_int_const_b.args])
    .subs(gams2_to_1_subs)
    .expand().collect(d[5], factor)
)

a_0_eq_bar_c = Eq(
    a_0_eq_bar_rho.lhs
    .subs([
        rhobar_sqrd_as_uvj.args, 
        rhobar_uvj_balanced.args
    ])
    .expand().collect([ubar(z, mu[1])*vbar(z, mu[1]), ubar(z, mu[2])*vbar(z, mu[2])], factor),
    a_0_eq_bar_rho.rhs
    .subs([
        h_uvbar12_no_const.args
    ])
    .subs(gams2_to_1_subs)
)

a_0_eq_bar_d = Eq(
    - (a_0_eq_bar_c.rhs - a_0_eq_bar_c.lhs).subs([(ubar(z, mu[1]),0), (ubar(z, mu[2]),0)]),
    (a_0_eq_bar_c.rhs - a_0_eq_bar_c.lhs) 
    - (a_0_eq_bar_c.rhs - a_0_eq_bar_c.lhs).subs([(ubar(z, mu[1]),0), (ubar(z, mu[2]),0)])
)
a_0_eq_bar_d = Eq(
    a_0_eq_bar_d.lhs*c[0], 
    sum(
        (_*c[0])
#         (_*c[0]).subs([gamma_lamda_bj_gam1.args]).factor() 
        for _ in a_0_eq_bar_d.rhs.args
    )
)

# In terms of h

a_0_eq_h = Eq(
    a_0_eq_bar_rho.lhs
    .subs([rhobar_as_h.args, gammabar_1_to_gamma_1.args])
    .expand().collect(h(z), factor),
    a_0_eq_bar_rho.rhs
    .subs([rhobar_as_h.args, gammabar_1_to_gamma_1.args])
    .factor()
)


a_0_eq_bar_rho
rhobar_uvj_balanced
rhobar_sqrd_as_uvj
print()
a_0_eq_bar_d
print()
a_0_eq_h

Eq(4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*rhobar(z)**2*lamda[1]**2 + (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*rhobar(z)*d[5]*lamda[1]**2/((gamma[1] - lamda[1])*(gamma[1] + lamda[1])) + (b[0]*lamda[1]**2 + b[1]*gamma[1]**2*lamda[1] + b[2]*gamma[1]**4)*d[5]**2*lamda[1]**2/(4*(gamma[1] - lamda[1])**2*(gamma[1] + lamda[1])**2), 2*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*h(z)*lamda[1])

Eq(rhobar(z), -ubar(z, mu[1])*vbar(z, mu[1])/2 - ubar(z, mu[2])*vbar(z, mu[2])/2)

Eq(rhobar(z)**2, (ubar(z, mu[1])**2*vbar(z, mu[1])**2 + ubar(z, mu[2])**2*vbar(z, mu[2])**2)/2 - d[5]**2*gamma[1]**2/(16*(gamma[1] - lamda[1])**2*(gamma[1] + lamda[1])**2))

Eq(-(b[0] - b[2]*gamma[1]**2)*c[0]*d[5]**2*lamda[1]**2/(4*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])), (ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(-2*(-gamma[1] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1] + 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1])*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*c[0]*lamda[1]/gamma[1] - 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[1])**2*vbar(z, mu[1])**2*c[0]*lamda[1]**2 - 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[2])**2*vbar(z, mu[2])**2*c[0]*lamda[1]**2 + (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])) + (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(-(b[1] + 2*b[2]*lamda[1])*h(z)*d[5]*lamda[1]/2 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*h(z)**2 + b[2]*d[5]**2*lamda[1]**2/4, 2*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*h(z)*lamda[1])

In [19]:
# One over h is a useful auxilary for transforming diffs
a_0_eq_on_over_h = Eq(
    0,
    sum(_/h(z) for _ in (a_0_eq_h.rhs - a_0_eq_h.lhs).args)
)
_ooh_coeff = a_0_eq_on_over_h.rhs.coeff(h(z),-1)
a_0_eq_on_over_h = Eq(
    -(a_0_eq_on_over_h.lhs - _ooh_coeff/h(z))/_ooh_coeff, 
    -sum(_/_ooh_coeff for _ in (a_0_eq_on_over_h.rhs - _ooh_coeff/h(z)).args)
)

a_0_eq_on_over_h

Eq(1/h(z), 8*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/(b[2]*d[5]**2*lamda[1]) + 2*(b[1] + 2*b[2]*lamda[1])/(b[2]*d[5]*lamda[1]) - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*h(z)/(b[2]*d[5]**2*lamda[1]**2))

We can look to see what the differential equation system would be for this Hamiltonian:

In [20]:
_u_v_bars = [ubar(z, mu[1]), ubar(z, mu[2]), vbar(z, mu[1]), vbar(z, mu[2])]
three_term_products = [prod(c) for c in combinations_with_replacement(_u_v_bars, 3)]

du_dv_bars_from_ham = [
    Eq(
        diff(ubar(z, mu[1]),z), 
        diff(a_0_eq_bar_d.rhs, vbar(z, mu[1]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(ubar(z, mu[2]),z), 
        diff(a_0_eq_bar_d.rhs, vbar(z, mu[2]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(vbar(z, mu[1]),z), 
        -diff(a_0_eq_bar_d.rhs, ubar(z, mu[1]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(vbar(z, mu[2]),z), 
        -diff(a_0_eq_bar_d.rhs, ubar(z, mu[2]))
        .expand().collect(three_term_products, factor)
    )
]
du_dv_bars_from_ham_subs = [_.args for _ in du_dv_bars_from_ham]

for _ in du_dv_bars_from_ham:
    _

Eq(Derivative(ubar(z, mu[1]), z), 2*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*ubar(z, mu[1])**2*ubar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] + 4*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*ubar(z, mu[1])*vbar(z, mu[1])*vbar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] + 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])**2*c[0]*lamda[1]**2/gamma[1] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[1])**2*vbar(z, mu[1])*c[0]*lamda[1]**2 + (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*ubar(z, mu[1])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(Derivative(ubar(z, mu[2]), z), 2*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*ubar(z, mu[1])*vbar(z, mu[1])**2*c[0]*lamda[1]**2/gamma[1] + 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])**2*c[0]*lamda[1]**2/gamma[1] + 4*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[1])*vbar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[2])**2*vbar(z, mu[2])*c[0]*lamda[1]**2 + (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*ubar(z, mu[2])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(Derivative(vbar(z, mu[1]), z), -4*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*ubar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[1])*c[0]*lamda[1]**2/gamma[1] - 2*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*vbar(z, mu[1])**2*vbar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] - 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*ubar(z, mu[2])**2*vbar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] + 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[1])*vbar(z, mu[1])**2*c[0]*lamda[1]**2 - (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*vbar(z, mu[1])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(Derivative(vbar(z, mu[2]), z), -2*sqrt(-gamma[1] - lamda[1])*(gamma[1] - lamda[1])**(3/2)*ubar(z, mu[1])**2*vbar(z, mu[1])*c[0]*lamda[1]**2/gamma[1] - 4*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[2])*c[0]*lamda[1]**2/gamma[1] - 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(gamma[1] + lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])**2*c[0]*lamda[1]**2/gamma[1] + 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*ubar(z, mu[2])*vbar(z, mu[2])**2*c[0]*lamda[1]**2 - (2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1])*vbar(z, mu[2])*c[0]*d[5]*lamda[1]**2/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

In [21]:
duv_bars_from_ham = [
    Eq(
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z),
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z)
        .doit()
        .subs(du_dv_bars_from_ham_subs)
        .factor()
    )
    for _j in range(2)
]

for _ in duv_bars_from_ham:
    _

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), -2*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(ubar(z, mu[1])*vbar(z, mu[1])*gamma[1] - ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] + ubar(z, mu[2])*vbar(z, mu[2])*gamma[1] + ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])*c[0]*lamda[1]**2/gamma[1])

Eq(Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), -2*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*(ubar(z, mu[1])*vbar(z, mu[1])*gamma[1] - ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] + ubar(z, mu[2])*vbar(z, mu[2])*gamma[1] + ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])*c[0]*lamda[1]**2/gamma[1])

### Transforming the system of differential equations to bar coordinates

We now transform the system of differential equations to bar coordinates. We remind ourselves of the original system:

In [184]:
dlog_uhatj = Eq(duhatj.lhs/uhat(z, mu[j]), sum(_/uhat(z, mu[j]) for _ in duhatj.rhs.args))
dlog_vhatj = Eq(dvhatj.lhs/vhat(z, mu[j]), sum(_/vhat(z, mu[j]) for _ in dvhatj.rhs.args))

uprod_hat_to_bar = Eq(
    Product(uhat(z, mu[k]), (k,1,2)), 
    Product(
        uhat(z, mu[k]).replace(*uj_as_ubarj_h.subs(j,k).args), 
        (k,1,2)
    )
    .doit()
    .subs(varsigma**2, 1)
    *(Product(ubar(z, mu[k]), (k,1,2))/Product(ubar(z, mu[k]), (k,1,2)).doit())
)
vprod_hat_to_bar = Eq(
    Product(vhat(z, mu[k]), (k,1,2)), 
    Product(
        vhat(z, mu[k]).replace(*vj_as_vbarj_h.subs(j,k).args), 
        (k,1,2)
    )
    .doit()
    .subs(varsigma**2, 1)
    *(Product(vbar(z, mu[k]), (k,1,2))/Product(vbar(z, mu[k]), (k,1,2)).doit())
)

u_v_prod_hat_bar_subs = [
    uprod_hat_to_bar.args,
    vprod_hat_to_bar.args
]
u_v_j_hat_to_bar_subs = [
    uj_as_ubarj_h.args,
    vj_as_vbarj_h.args
]

dlog_ubarj_b = Eq(
    dlog_uhatj.lhs
    .subs([uj_as_ubarj_h.args])
    .doit().expand()
    + diff(log(h(z)), z)/2,
    dlog_uhatj.rhs
    .subs(u_v_prod_hat_bar_subs)
    .subs(u_v_j_hat_to_bar_subs)
    .subs(varsigma**2, 1)
    + diff(log(h(z)), z)/2
)
dlog_vbarj_b = Eq(
    dlog_vhatj.lhs
    .subs([vj_as_vbarj_h.args])
    .doit().expand()
    + diff(log(h(z)), z)/2,
    dlog_vhatj.rhs
    .subs(u_v_prod_hat_bar_subs)
    .subs(u_v_j_hat_to_bar_subs)
    .subs(varsigma**2, 1)
    + diff(log(h(z)), z)/2
)

du_barj_trns = Eq(
    dlog_ubarj_b.lhs*ubar(z, mu[j]),
    sum(_*ubar(z, mu[j]) for _ in dlog_ubarj_b.rhs.args)
)
dv_barj_trns = Eq(
    dlog_vbarj_b.lhs*vbar(z, mu[j]),
    sum(_*vbar(z, mu[j]) for _ in dlog_vbarj_b.rhs.args)
)

du_dv_bar_from_trns_subs = [
    du_barj_trns.subs(j,1).doit().args,
    du_barj_trns.subs(j,2).doit().args,
    dv_barj_trns.subs(j,1).doit().args,
    dv_barj_trns.subs(j,2).doit().args
]



duhatj
dvhatj
dlog_uhatj
dlog_vhatj
uprod_hat_to_bar
vprod_hat_to_bar
dlog_ubarj_b
dlog_vbarj_b
print()
du_barj_trns
dv_barj_trns

Eq(Derivative(uhat(z, mu[j]), z), -uhat(z, mu[j])**2*vhat(z, mu[j])*b[2] + uhat(z, mu[j])*b[1]/2 + Product(vhat(z, mu[k]), (k, 1, 2))/vhat(z, mu[j]))

Eq(Derivative(vhat(z, mu[j]), z), uhat(z, mu[j])*vhat(z, mu[j])**2*b[2] - vhat(z, mu[j])*b[1]/2 - Product(uhat(z, mu[k]), (k, 1, 2))/uhat(z, mu[j]))

Eq(Derivative(uhat(z, mu[j]), z)/uhat(z, mu[j]), -uhat(z, mu[j])*vhat(z, mu[j])*b[2] + b[1]/2 + Product(vhat(z, mu[k]), (k, 1, 2))/(uhat(z, mu[j])*vhat(z, mu[j])))

Eq(Derivative(vhat(z, mu[j]), z)/vhat(z, mu[j]), uhat(z, mu[j])*vhat(z, mu[j])*b[2] - b[1]/2 - Product(uhat(z, mu[k]), (k, 1, 2))/(uhat(z, mu[j])*vhat(z, mu[j])))

Eq(Product(uhat(z, mu[k]), (k, 1, 2)), 2*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*lamda[1]*Product(ubar(z, mu[k]), (k, 1, 2))/h(z))

Eq(Product(vhat(z, mu[k]), (k, 1, 2)), 2*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*lamda[1]*Product(vbar(z, mu[k]), (k, 1, 2))/h(z))

Eq(Derivative(ubar(z, mu[j]), z)/ubar(z, mu[j]), sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*Product(vbar(z, mu[k]), (k, 1, 2))/((gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])) - 2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*b[2]*lamda[1]/h(z) + b[1]/2 + Derivative(h(z), z)/(2*h(z)))

Eq(Derivative(vbar(z, mu[j]), z)/vbar(z, mu[j]), -sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*Product(ubar(z, mu[k]), (k, 1, 2))/((gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])) + 2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*b[2]*lamda[1]/h(z) - b[1]/2 + Derivative(h(z), z)/(2*h(z)))

Eq(Derivative(ubar(z, mu[j]), z), sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*Product(vbar(z, mu[k]), (k, 1, 2))/((gamma[j] - lamda[1])*vbar(z, mu[j])) - 2*(gamma[j] - lamda[1])*ubar(z, mu[j])**2*vbar(z, mu[j])*b[2]*lamda[1]/h(z) + ubar(z, mu[j])*b[1]/2 + ubar(z, mu[j])*Derivative(h(z), z)/(2*h(z)))

Eq(Derivative(vbar(z, mu[j]), z), -sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*Product(ubar(z, mu[k]), (k, 1, 2))/((gamma[j] - lamda[1])*ubar(z, mu[j])) + 2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])**2*b[2]*lamda[1]/h(z) - vbar(z, mu[j])*b[1]/2 + vbar(z, mu[j])*Derivative(h(z), z)/(2*h(z)))

We define our familiar auxillary constants:

In [23]:
gamma_j_prod_b_poly = Eq(
    Product(gamma[j] -  lamda[1], (j,1,2)), 
    (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4
)
C_gam_prod = Eq(Product(sqrt(gamma[j] - lamda[1]), (j,1,2)), C)
C_gam_prod_sqrd = Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)
C_gam_prod_sign = Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m * C)
chi_d5_C = Eq(chi, 16*(-1)**m*C/d[5])
C_d5_chi = Eq(C, solve(chi_d5_C,C)[0])

b_C_one_over_chi = Eq(
    (
        d5_dj.rhs
        .subs([_.args for _ in d_j_coeffs])
        .subs([_.args for _ in c_j_coeffs])
        .subs(b[0], solve(C_gam_prod_sign, b[0])[0])
        /(16*(-1)**m*C)
    )
    .expand(),
    (d5_dj.lhs/(16*(-1)**m*C)).subs([C_d5_chi.args])
)

d5_C_b_lam = Eq(
    d5_dj.lhs,
    d5_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([(b[0], solve(C_gam_prod_sign, b[0])[0])])
    .expand()
    .collect(C, factor)
)

__tb = 2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1]
b_long_C = Eq(
    __tb,
    __tb
    .subs([
        gamma_lamda_bj_gam1.args, 
        (C_gam_prod_sqrd.rhs, C_gam_prod_sqrd.lhs)
    ])
    .expand().collect(C, factor)
    .subs([
        C_gam_prod_sign.args,
        (b[1], solve(b_C_one_over_chi, b[1])[0])
    ])
    .expand().collect(C, factor)
    .subs((-1)**(2*m),1)
)

gam_prod_C_sqrd = Eq(
    (gamma[1]**2 - lamda[1]**2)
    .factor(), 
    (gamma[1]**2 - lamda[1]**2)
    .subs([gamma_lamda_bj_gam1.args, (C_gam_prod_sqrd.rhs, C_gam_prod_sqrd.lhs)])
)

gam1_sqrd_chi = Eq(
    gamma[1]**2, 
    solve(gam_prod_C_sqrd.expand().subs([C_d5_chi.args, ((-1)**(2*m), 1)]), gamma[1]**2)[0]
)

gamma_j_prod_b_poly
C_gam_prod
C_gam_prod_sqrd
C_gam_prod_sign
chi_d5_C
C_d5_chi
b_C_one_over_chi
d5_C_b_lam
b_long_C
gam_prod_C_sqrd
gam1_sqrd_chi

Eq(Product(gamma[j] - lamda[1], (j, 1, 2)), (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(Product(sqrt(gamma[j] - lamda[1]), (j, 1, 2)), C)

Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m*C)

Eq(chi, 16*(-1)**m*C/d[5])

...

Eq(b[1]/4 + b[2]*lamda[1]/2 - lamda[1]/(2*(-1)**m*C), 1/chi)

Eq(d[5], 4*(-1)**m*C*(b[1] + 2*b[2]*lamda[1]) - 8*lamda[1])

Eq(2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1], -4*C**2/chi + 2*C*lamda[1]/(-1)**m)

Eq((gamma[1] - lamda[1])*(gamma[1] + lamda[1]), -C**2)

Eq(gamma[1]**2, -chi**2*d[5]**2/256 + lamda[1]**2)

Then recast the differential equations in terms of these:

In [24]:
_b2_chi = Eq(b[2], solve(b_C_one_over_chi, b[2])[0])
_uvj_chi_term = -2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*b[2]*lamda[1]/h(z)
uvj_bar_ov_h_term = Eq(
    _uvj_chi_term.factor(),
    _uvj_chi_term
    .subs([
        uvbarj_as_h.args
    ])
    .expand().collect(h(z), factor)
    .subs([
        (
            a_0_eq_on_over_h.lhs*b[2]*d[5]*lamda[1],
            sum(_*b[2]*d[5]*lamda[1] for _ in a_0_eq_on_over_h.rhs.args)
        ),
        C_gam_prod.doit().args,
        C_gam_prod_sign.args,
        C_d5_chi.args,
        h_as_uvbarj.args,
        _b2_chi.args,
        C_d5_chi.args
    ])
    .expand().collect([(-1)**m, (-1)**m*chi, ubar(z, mu[j])*vbar(z, mu[j]), b[1], b[2]], factor)
    .subs([
        (gamma[j]**2, solve(gam_prod_C_sqrd.expand().subs([C_d5_chi.args, ((-1)**(2*m),1)]), gamma[1]**2)[0])
    ])
    .expand().collect([(-1)**m, (-1)**m*chi, ubar(z, mu[j])*vbar(z, mu[j]), b[1], b[2]], factor)
)

du_barj_trns_b = Eq(
    du_barj_trns.lhs,
    du_barj_trns.rhs
    .subs([
        dlog_h_as_u_v_bar.args,
        (
            uvj_bar_ov_h_term.lhs*ubar(z, mu[j]), 
            sum(_*ubar(z, mu[j]) for _ in uvj_bar_ov_h_term.rhs.args)
        ),
        C_gam_prod.doit().subs([C_d5_chi.args]).args
    ])
)
dv_barj_trns_b = Eq(
    dv_barj_trns.lhs,
    dv_barj_trns.rhs
    .subs([
        dlog_h_as_u_v_bar.args,
        (
            -uvj_bar_ov_h_term.lhs*vbar(z, mu[j]), 
            -sum(_*vbar(z, mu[j]) for _ in uvj_bar_ov_h_term.rhs.args)
        ),
        C_gam_prod.doit().subs([C_d5_chi.args]).args
    ])
)


dlog_h_as_u_v_bar
uvj_bar_ov_h_term
print()
du_barj_trns_b
print()
dv_barj_trns_b

Eq(Derivative(h(z), z)/h(z), 4*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(-2*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])*b[2]*lamda[1]/h(z), chi*(d[5] - 8*lamda[1])*d[5]/(128*(gamma[j] - lamda[1])*lamda[1]) + chi*ubar(z, mu[j])*vbar(z, mu[j])/2 + (gamma[j] - lamda[1])*b[1]/(2*lamda[1]) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))/(4*(-1)**m))

Eq(Derivative(ubar(z, mu[j]), z), chi*(d[5] - 8*lamda[1])*ubar(z, mu[j])*d[5]/(128*(gamma[j] - lamda[1])*lamda[1]) + chi*ubar(z, mu[j])**2*vbar(z, mu[j])/2 + (gamma[j] - lamda[1])*ubar(z, mu[j])*b[1]/(2*lamda[1]) + ubar(z, mu[j])*b[1]/2 + chi*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*ubar(z, mu[j])/(8*(-1)**m) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*ubar(z, mu[j])/(4*(-1)**m) + chi*d[5]*Product(vbar(z, mu[k]), (k, 1, 2))/(16*(-1)**m*(gamma[j] - lamda[1])*vbar(z, mu[j])))

Eq(Derivative(vbar(z, mu[j]), z), -chi*(d[5] - 8*lamda[1])*vbar(z, mu[j])*d[5]/(128*(gamma[j] - lamda[1])*lamda[1]) - chi*ubar(z, mu[j])*vbar(z, mu[j])**2/2 - (gamma[j] - lamda[1])*vbar(z, mu[j])*b[1]/(2*lamda[1]) - vbar(z, mu[j])*b[1]/2 + chi*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*vbar(z, mu[j])/(8*(-1)**m) + chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*vbar(z, mu[j])/(4*(-1)**m) - chi*d[5]*Product(ubar(z, mu[k]), (k, 1, 2))/(16*(-1)**m*(gamma[j] - lamda[1])*ubar(z, mu[j])))

We have experimented with trying to guess the canonical Hamiltonian that would produce the above differential equations but it is proving difficult to reproduce the wave mixing terms as written.

### Normalising the transformed Hamiltonian

In [25]:
duv_bars_from_trns = [
    Eq(
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z),
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z)
        .doit()
        .subs(du_dv_bar_from_trns_subs)
        .expand().collect(diff(h(z),z), factor)
    )
    for _j in range(2)
]
duv_bars_from_trns_b = [
    Eq(
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z),
        Derivative(ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),z)
        .doit()
        .subs(du_dv_bar_from_trns_subs)
        .expand().collect(diff(h(z),z), factor)
        .subs([dlog_h_as_u_v_bar.args])
        .factor()
    )
    for _j in range(2)
]

for _ in duv_bars_from_trns:
    _
print()  
for _ in duv_bars_from_trns_b:
    _

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), -(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[2] - lamda[1])/sqrt(gamma[1] - lamda[1]) + ubar(z, mu[1])*vbar(z, mu[1])*Derivative(h(z), z)/h(z))

Eq(Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), -(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])/sqrt(gamma[2] - lamda[1]) + ubar(z, mu[2])*vbar(z, mu[2])*Derivative(h(z), z)/h(z))

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), -(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[2] - lamda[1])*(-4*ubar(z, mu[1])*vbar(z, mu[1])*gamma[1] + 4*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] + d[5])/(sqrt(gamma[1] - lamda[1])*d[5]))

Eq(Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), -(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*sqrt(gamma[1] - lamda[1])*(-4*ubar(z, mu[2])*vbar(z, mu[2])*gamma[2] + 4*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1] + d[5])/(sqrt(gamma[2] - lamda[1])*d[5]))

We confirm that the transformed Hamiltonian can match this intermodal power evolution in the transformed system with the following choice for $c_0$:

In [26]:
c0_scale = Eq(
    c[0], 
    solve(
        Eq(
            1,
           (duv_bars_from_ham[0].rhs/duv_bars_from_trns_b[0].rhs)
            .subs([uvbarj_as_h.subs(j,1).args, uvbarj_as_h.subs(j,2).args])
            .subs(gams2_to_1_subs)
            .factor()
        ),
        c[0]
    )[0]
)

c0_scale

Eq(c[0], -1/(d[5]*lamda[1]**2))

Then we update the transformed Hamiltonian accordingly:

In [27]:
a_0_eq_bar_e = a_0_eq_bar_d.subs([
    c0_scale.args,
    C_gam_prod.doit().subs(gams2_to_1_subs).args,
    C_gam_prod_sign.args,
    b_long_C.args,
    gam_prod_C_sqrd.args,
    (d[5], solve(C_d5_chi, d[5])[0]),
    (gamma[1]**2, solve(gam_prod_C_sqrd.expand(), gamma[1]**2)[0])
])
a_0_eq_bar_e = Eq(
    a_0_eq_bar_e.lhs.expand().collect(C, factor),
    a_0_eq_bar_e.rhs
    .subs([
        (
            a_0_eq_bar_e.rhs.coeff(ubar(z, mu[1])*vbar(z, mu[1])), 
            a_0_eq_bar_e.rhs.coeff(ubar(z, mu[1])*vbar(z, mu[1])).expand()
        )
    ])
)

du_dv_bars_from_ham_chi = [
    Eq(
        diff(ubar(z, mu[1]),z), 
        diff(a_0_eq_bar_e.rhs, vbar(z, mu[1]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(ubar(z, mu[2]),z), 
        diff(a_0_eq_bar_e.rhs, vbar(z, mu[2]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(vbar(z, mu[1]),z), 
        -diff(a_0_eq_bar_e.rhs, ubar(z, mu[1]))
        .expand().collect(three_term_products, factor)
    ),
    Eq(
        diff(vbar(z, mu[2]),z), 
        -diff(a_0_eq_bar_e.rhs, ubar(z, mu[2]))
        .expand().collect(three_term_products, factor)
    )
]
du_dv_bars_from_ham_chi_subs = [_.args for _ in du_dv_bars_from_ham_chi]

dlogu_dlogv_bars_from_ham_chi = [
    Eq(_d.lhs/_f, sum(_/_f for _ in _d.rhs.args))
    for _f, _d in zip(_u_v_bars, du_dv_bars_from_ham_chi)
]

a_0_eq_bar_e 
# print()
# for _ in du_dv_bars_from_ham_chi:
#     _
print()
for _ in dlogu_dlogv_bars_from_ham_chi:
    _

Eq(-4*(-1)**m*C*b[2]/chi - 4*(-1)**m*(b[0] - b[2]*lamda[1]**2)/(C*chi), chi*ubar(z, mu[1])**2*vbar(z, mu[1])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])**2/4 + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[1])*vbar(z, mu[1]) + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[2])*vbar(z, mu[2]) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(-2*(-gamma[1] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1] + 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1])/(16*(-1)**m*gamma[1]*lamda[1]))

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), chi*ubar(z, mu[1])*vbar(z, mu[1])/2 - chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])/(8*(-1)**m*gamma[1]) - chi*(gamma[1] - lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])/(4*(-1)**m*gamma[1]) - chi*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])**2/(8*(-1)**m*ubar(z, mu[1])*gamma[1]) - (2*(-1)**m*C - chi*lamda[1])/((-1)**m*C*chi))

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), chi*ubar(z, mu[2])*vbar(z, mu[2])/2 - chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])**2/(8*(-1)**m*ubar(z, mu[2])*gamma[1]) - chi*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])/(8*(-1)**m*gamma[1]) - chi*(gamma[1] + lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])/(4*(-1)**m*gamma[1]) - (2*(-1)**m*C - chi*lamda[1])/((-1)**m*C*chi))

Eq(Derivative(vbar(z, mu[1]), z)/vbar(z, mu[1]), -chi*ubar(z, mu[1])*vbar(z, mu[1])/2 + chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])/(4*(-1)**m*gamma[1]) + chi*(gamma[1] - lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])/(8*(-1)**m*gamma[1]) + chi*(gamma[1] + lamda[1])*ubar(z, mu[2])**2*vbar(z, mu[2])/(8*(-1)**m*vbar(z, mu[1])*gamma[1]) + (2*(-1)**m*C - chi*lamda[1])/((-1)**m*C*chi))

Eq(Derivative(vbar(z, mu[2]), z)/vbar(z, mu[2]), -chi*ubar(z, mu[2])*vbar(z, mu[2])/2 + chi*(gamma[1] - lamda[1])*ubar(z, mu[1])**2*vbar(z, mu[1])/(8*(-1)**m*vbar(z, mu[2])*gamma[1]) + chi*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])/(4*(-1)**m*gamma[1]) + chi*(gamma[1] + lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])/(8*(-1)**m*gamma[1]) + (2*(-1)**m*C - chi*lamda[1])/((-1)**m*C*chi))

We return to look at the phase parts of the transformed system. We have two system one from the transformed Hamiltonian and one form the transformed differential equations for which we do not know the canonical Hamiltonian. We compare the two differential equation systems and we see that they differ by a gauge transform: 

In [28]:
dlog_phase_trns_j = Eq(
    (dlog_ubarj_b.lhs - dlog_vbarj_b.lhs)/2,
    ((dlog_ubarj_b.rhs - dlog_vbarj_b.rhs)/2)
    .collect([b[1], b[2]], factor)
    .subs([
        C_gam_prod.doit().args,
        C_d5_chi.args
    ])
)

dlog_phase_trns_j = dlog_phase_trns_j.subs([uvj_bar_ov_h_term.args])


# Corresponding equations from Hamiltonian
dlog_phase_ham_1 = Eq(
    (dlogu_dlogv_bars_from_ham_chi[0].lhs - dlogu_dlogv_bars_from_ham_chi[2].lhs)/2,
    ((dlogu_dlogv_bars_from_ham_chi[0].rhs - dlogu_dlogv_bars_from_ham_chi[2].rhs)/2)
    .collect([chi/gamma[1]], factor)
)
dlog_phase_ham_2 = Eq(
    (dlogu_dlogv_bars_from_ham_chi[1].lhs - dlogu_dlogv_bars_from_ham_chi[3].lhs)/2,
    ((dlogu_dlogv_bars_from_ham_chi[1].rhs - dlogu_dlogv_bars_from_ham_chi[3].rhs)/2)
    .collect([chi/gamma[1]], factor)
)
_split_fwm_term_1 = Eq(
    dlog_phase_ham_1.rhs.args[1],
    dlog_phase_ham_1.rhs.args[1]
    .subs(uv_bars_to_rhobar_subs)
    .subs(gams2_to_1_subs)
    .subs(rhobar(z), solve(Eq(*uv_bars_to_rhobar_subs[0]), rhobar(z))[0])
    .expand().collect(ubar(z, mu[1])*vbar(z, mu[1]), factor) 
    .subs(ubar(z, mu[1])*vbar(z, mu[1]), f(z)) 
    .expand().collect(f(z), factor)
    .subs([(f(z), ubar(z, mu[1])*vbar(z, mu[1])), gammabar_1_to_gamma_1.args])  
)
_split_fwm_term_2 = Eq(
    dlog_phase_ham_2.rhs.args[1],
    dlog_phase_ham_2.rhs.args[1]
    .subs(uv_bars_to_rhobar_subs)
    .subs(gams2_to_1_subs)
    .subs(rhobar(z), solve(Eq(*uv_bars_to_rhobar_subs[1]), rhobar(z))[0])
    .expand().collect(ubar(z, mu[2])*vbar(z, mu[2]), factor) 
    .subs(ubar(z, mu[2])*vbar(z, mu[2]), f(z)) 
    .subs(gams2_to_1_subs)
    .expand().collect(f(z), factor)
    .subs([(f(z), ubar(z, mu[2])*vbar(z, mu[2])), gammabar_1_to_gamma_1.args])  
)
_split_pm_term_1 = Eq(
    dlog_phase_ham_1.rhs.args[0],
    dlog_phase_ham_1.rhs.args[0]
    .expand().collect(chi, factor)
)
_split_pm_term_2 = Eq(
    dlog_phase_ham_2.rhs.args[0],
    dlog_phase_ham_2.rhs.args[0]
    .expand().collect(chi, factor)
)
dlog_phase_ham_1 = dlog_phase_ham_1.subs([_split_fwm_term_1.args, _split_pm_term_1.args])
dlog_phase_ham_2 = dlog_phase_ham_2.subs([_split_fwm_term_2.args, _split_pm_term_2.args])




print('From the transformed differential equations:')
dlog_phase_trns_j
print('From the transformed Hamiltonian:')
dlog_phase_ham_1
dlog_phase_ham_2

From the transformed differential equations:


Eq(-Derivative(vbar(z, mu[j]), z)/(2*vbar(z, mu[j])) + Derivative(ubar(z, mu[j]), z)/(2*ubar(z, mu[j])), chi*(d[5] - 8*lamda[1])*d[5]/(128*(gamma[j] - lamda[1])*lamda[1]) + chi*ubar(z, mu[j])*vbar(z, mu[j])/2 + (gamma[j] - lamda[1])*b[1]/(2*lamda[1]) + b[1]/2 - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))/(4*(-1)**m) + chi*(Product(ubar(z, mu[k]), (k, 1, 2)) + Product(vbar(z, mu[k]), (k, 1, 2)))*d[5]/(32*(-1)**m*(gamma[j] - lamda[1])*ubar(z, mu[j])*vbar(z, mu[j])))

From the transformed Hamiltonian:


Eq(-Derivative(vbar(z, mu[1]), z)/(2*vbar(z, mu[1])) + Derivative(ubar(z, mu[1]), z)/(2*ubar(z, mu[1])), chi*ubar(z, mu[1])*vbar(z, mu[1])/2 - 2/chi - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(2*gamma[1] - lamda[1])/(8*(-1)**m*gamma[1]) + chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*d[5]/(32*(-1)**m*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])) + lamda[1]/((-1)**m*C))

Eq(-Derivative(vbar(z, mu[2]), z)/(2*vbar(z, mu[2])) + Derivative(ubar(z, mu[2]), z)/(2*ubar(z, mu[2])), chi*ubar(z, mu[2])*vbar(z, mu[2])/2 - 2/chi - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(2*gamma[1] + lamda[1])/(8*(-1)**m*gamma[1]) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*d[5]/(32*(-1)**m*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])) + lamda[1]/((-1)**m*C))

In [142]:
dphi_1_trns_ham = Eq(
    diff(phi(z, 1),z),
    (dlog_phase_ham_1.rhs - dlog_phase_trns_j.subs(j,1).doit().rhs)
    .subs([C_d5_chi.args])
    .expand().collect([chi*(-1)**(-m), chi], factor)
)

dphi_2_trns_ham = Eq(
    diff(phi(z, 2),z),
    (dlog_phase_ham_2.rhs - dlog_phase_trns_j.subs(j,2).doit().rhs)
    .subs(gams2_to_1_subs)
    .subs([C_d5_chi.args])
    .expand().collect([chi*(-1)**(-m), chi], factor)
)

_const_diff = (
    dphi_1_trns_ham.rhs.subs([(ubar(z,mu[1]),0), (vbar(z,mu[1]),0), (b[1],0)]) - 
    dphi_2_trns_ham.rhs.subs([(ubar(z,mu[1]),0), (vbar(z,mu[1]),0), (b[1],0)])
)
_const_sum = (
    dphi_1_trns_ham.rhs.subs([(ubar(z,mu[1]),0), (vbar(z,mu[1]),0), (b[1],0)]) + 
    dphi_2_trns_ham.rhs.subs([(ubar(z,mu[1]),0), (vbar(z,mu[1]),0), (b[1],0)])
)

const_diff_subs = [
    (
        (_const_diff + _const_sum)/2,
        (
            _const_diff.simplify().subs([gam1_sqrd_chi.args]).simplify() + 
            _const_sum.simplify().subs([gam1_sqrd_chi.args]).simplify()
        )/2
    ),
    (
        -(_const_diff - _const_sum)/2,
        -(
            _const_diff.simplify().subs([gam1_sqrd_chi.args]).simplify() - 
            _const_sum.simplify().subs([gam1_sqrd_chi.args]).simplify()
        )/2
    )
]



dphi_1_trns_ham = dphi_1_trns_ham.subs(const_diff_subs)
dphi_2_trns_ham = dphi_2_trns_ham.subs(const_diff_subs)

dphi_1_trns_ham
dphi_2_trns_ham

Eq(
    dphi_1_trns_ham.lhs + dphi_2_trns_ham.lhs, 
    dphi_1_trns_ham.rhs + dphi_2_trns_ham.rhs
)

Eq(Derivative(phi(z, 1), z), -b[1]*gamma[1]/(2*lamda[1]) + 2*(d[5] - 8*lamda[1])*gamma[1]/(chi*d[5]*lamda[1]) + chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*lamda[1]/(8*(-1)**m*gamma[1]))

Eq(Derivative(phi(z, 2), z), b[1]*gamma[1]/(2*lamda[1]) - 2*(d[5] - 8*lamda[1])*gamma[1]/(chi*d[5]*lamda[1]) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*lamda[1]/(8*(-1)**m*gamma[1]))

Eq(Derivative(phi(z, 1), z) + Derivative(phi(z, 2), z), 0)

We thus ask what the integral of this phase term is:

In [143]:
a_0_eq_on_over_h_b = (
    a_0_eq_on_over_h    
    .subs(gams2_to_1_subs)
    .subs([
        C_gam_prod.doit().subs(gams2_to_1_subs).args,
        C_gam_prod_sign.args,
        C_d5_chi.args
    ])
)
_wmt = a_0_eq_on_over_h_b.rhs.subs(chi*(-1)**(-m), 0)
_ccc = 2*b[2]*d[5]*lamda[1]**2/gamma[1]

a_0_eq_on_over_h_b = Eq(
    (a_0_eq_on_over_h_b.rhs - _wmt)*_ccc,
    ((a_0_eq_on_over_h_b.lhs - _wmt)*_ccc)
    .expand()
    .subs([
        (b[2], solve(b_C_one_over_chi, b[2])[0]),
        C_d5_chi.args
    ]).collect(h(z), factor)
)

dphi_1_h = Eq(
    dphi_1_trns_ham.lhs,
    dphi_1_trns_ham.rhs
    .subs([a_0_eq_on_over_h_b.args])
    .collect([h(z), b[1]], factor)
    .subs([gam1_sqrd_chi.args])
    .collect([h(z), b[1]], factor)
    .subs([(b[1], solve(d5_C_b_lam, b[1])[0]), C_d5_chi.args])
    .expand()
    .collect([h(z), b[2]], factor)
    .subs([
        gam1_sqrd_chi.args
    ])
    .expand()
    .collect([h(z), b[2], lamda[1]], factor)
)

a_0_eq_on_over_h_b
dphi_1_h

Eq(chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*lamda[1]/((-1)**m*gamma[1]), chi*h(z)/gamma[1] - 16*(d[5] + 8*lamda[1])*lamda[1]/(chi*d[5]*gamma[1]) - (chi*b[1]*d[5] - 4*d[5] - 32*lamda[1])*lamda[1]/(chi*h(z)*gamma[1]))

Eq(Derivative(phi(z, 1), z), chi*h(z)/(8*gamma[1]) + chi*d[5]/(8*gamma[1]) + b[2]*gamma[1] + b[2]*d[5]*lamda[1]**2/(4*h(z)*gamma[1]) - 2*lamda[1]/(chi*gamma[1]) - 48*lamda[1]**2/(chi*d[5]*gamma[1]))

In [133]:
h_wp_z1_z = Eq(
    (-uvbar_12_d5_gam_wp_z1.lhs + h_u_v_bar.lhs - h_u_v_bar.rhs)
    .simplify(),
    (-uvbar_12_d5_gam_wp_z1.rhs)
    .simplify()
)

h_wp = Eq(
    chi*h(z)/8/gamma[1], 
    (chi*h(z)/8/gamma[1]).subs([h_wp_z1_z.args])
)
_b2_ov_h_term = b[2]*d[5]*lamda[1]**2/(4*h(z)*gamma[1])
one_ov_h_wp = Eq(
    _b2_ov_h_term,
    _b2_ov_h_term
    .subs([
        h_wp_z1_z.args,
        (d[5], solve(wppz1_djs, d[5])[0]),
        sqrt_d4_b2.args
    ])
)

h_wp_z1_z
h_wp
one_ov_h_wp

Eq(h(z), 2*(wp(z1, g2, g3) - wp(z - z0, g2, g3))*lamda[1])

Eq(chi*h(z)/(8*gamma[1]), chi*(wp(z1, g2, g3) - wp(z - z0, g2, g3))*lamda[1]/(4*gamma[1]))

Eq(b[2]*d[5]*lamda[1]**2/(4*h(z)*gamma[1]), -\wp'(z1, g2, g3)*lamda[1]/(2*(wp(z1, g2, g3) - wp(z - z0, g2, g3))*gamma[1]))

In [129]:
wp_mu_j_chi = Eq(
    wp_z1_z0_mu_j_d5_lam.lhs.subs(j,1) - wp_z1_z0_mu_j_d5_lam.lhs.subs(j,2),
    (wp_z1_z0_mu_j_d5_lam.rhs.subs(j,1) - wp_z1_z0_mu_j_d5_lam.rhs.subs(j,2))
    .subs(gams2_to_1_subs)
    .simplify()
    .subs([
        gam_prod_C_sqrd.args,
        C_d5_chi.args,
        ((-1)**(2*m),1)
    ])
)
wp_mu_j_z1_chi =Eq(
    wp_z1_z0_mu_j_d5_lam.lhs.subs(j,1) + wp_z1_z0_mu_j_d5_lam.lhs.subs(j,2),
    (wp_z1_z0_mu_j_d5_lam.rhs.subs(j,1) + wp_z1_z0_mu_j_d5_lam.rhs.subs(j,2))
    .simplify()
    .subs(gams2_to_1_subs)
    .simplify()
    .subs([
        gam_prod_C_sqrd.args,
        C_d5_chi.args,
        ((-1)**(2*m),1)
    ])
)

wp_gam1_lam = Eq(
    (wp_mu_j_chi.lhs - wp_mu_j_z1_chi.lhs)/2,
    ((wp_mu_j_chi.rhs - wp_mu_j_z1_chi.rhs)/2)
    .factor()
)
wp_gam2_lam = Eq(
    (wp_mu_j_chi.lhs + wp_mu_j_z1_chi.lhs)/2,
    ((wp_mu_j_chi.rhs + wp_mu_j_z1_chi.rhs)/2)
    .factor()
)

wp_mu_1_mu_2_chi = Eq(
    -wp_gam1_lam.lhs * wp_gam2_lam.lhs,
    -(wp_gam1_lam.rhs * wp_gam2_lam.rhs)
    .subs([
        gam_prod_C_sqrd.args,
        C_d5_chi.args,
        ((-1)**(2*m),1)
    ])
)


wp_mu_j_chi
wp_mu_j_z1_chi
wp_gam1_lam
wp_gam2_lam
wp_mu_1_mu_2_chi

Eq(-wp(-z0 + mu[1], g2, g3) + wp(-z0 + mu[2], g2, g3), 128*gamma[1]/(chi**2*d[5]))

Eq(2*wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3), 128*lamda[1]/(chi**2*d[5]))

Eq(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3), 64*(gamma[1] - lamda[1])/(chi**2*d[5]))

Eq(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3), 64*(gamma[1] + lamda[1])/(chi**2*d[5]))

Eq((wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3)), 16/chi**2)

## Solution of the differential equations from the Hamiltonian

In [156]:
a_0_eq_bar_e 
print()
for _ in du_dv_bars_from_ham_chi:
    _

Eq(-4*(-1)**m*C*b[2]/chi - 4*(-1)**m*(b[0] - b[2]*lamda[1]**2)/(C*chi), chi*ubar(z, mu[1])**2*vbar(z, mu[1])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])**2/4 + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[1])*vbar(z, mu[1]) + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[2])*vbar(z, mu[2]) - chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(-2*(-gamma[1] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*lamda[1] + 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*lamda[1])/(16*(-1)**m*gamma[1]*lamda[1]))

Eq(Derivative(ubar(z, mu[1]), z), chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - chi*(gamma[1] - lamda[1])*ubar(z, mu[1])**2*ubar(z, mu[2])/(8*(-1)**m*gamma[1]) - chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*vbar(z, mu[2])/(4*(-1)**m*gamma[1]) - chi*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])**2/(8*(-1)**m*gamma[1]) - (2*(-1)**m*C - chi*lamda[1])*ubar(z, mu[1])/((-1)**m*C*chi))

Eq(Derivative(ubar(z, mu[2]), z), chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])**2/(8*(-1)**m*gamma[1]) - chi*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])**2/(8*(-1)**m*gamma[1]) - chi*(gamma[1] + lamda[1])*ubar(z, mu[2])*vbar(z, mu[1])*vbar(z, mu[2])/(4*(-1)**m*gamma[1]) - (2*(-1)**m*C - chi*lamda[1])*ubar(z, mu[2])/((-1)**m*C*chi))

Eq(Derivative(vbar(z, mu[1]), z), -chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + chi*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[1])/(4*(-1)**m*gamma[1]) + chi*(gamma[1] - lamda[1])*vbar(z, mu[1])**2*vbar(z, mu[2])/(8*(-1)**m*gamma[1]) + chi*(gamma[1] + lamda[1])*ubar(z, mu[2])**2*vbar(z, mu[2])/(8*(-1)**m*gamma[1]) + (2*(-1)**m*C - chi*lamda[1])*vbar(z, mu[1])/((-1)**m*C*chi))

Eq(Derivative(vbar(z, mu[2]), z), -chi*ubar(z, mu[2])*vbar(z, mu[2])**2/2 + chi*(gamma[1] - lamda[1])*ubar(z, mu[1])**2*vbar(z, mu[1])/(8*(-1)**m*gamma[1]) + chi*(gamma[1] + lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[2])/(4*(-1)**m*gamma[1]) + chi*(gamma[1] + lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])**2/(8*(-1)**m*gamma[1]) + (2*(-1)**m*C - chi*lamda[1])*vbar(z, mu[2])/((-1)**m*C*chi))

In [161]:
duvbar1 = Eq(
    Derivative(ubar(z, mu[1])*vbar(z,mu[1]),z),
    Derivative(ubar(z, mu[1])*vbar(z,mu[1]),z)
    .doit()
    .subs(du_dv_bars_from_ham_chi_subs)
    .factor()
)

duvbar1

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), chi*(ubar(z, mu[1])*ubar(z, mu[2]) - vbar(z, mu[1])*vbar(z, mu[2]))*(ubar(z, mu[1])*vbar(z, mu[1])*gamma[1] - ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] + ubar(z, mu[2])*vbar(z, mu[2])*gamma[1] + ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])/(8*(-1)**m*gamma[1]))

In [178]:
a_0_eq_bar_wm_pm = Eq(
    -(a_0_eq_bar_e.rhs - a_0_eq_bar_e.rhs.subs(ubar(z, mu[1])*ubar(z,mu[2]) + vbar(z, mu[1])*vbar(z,mu[2]), 0))
    .factor(),
    -(a_0_eq_bar_e.lhs - a_0_eq_bar_e.rhs.subs(ubar(z, mu[1])*ubar(z,mu[2]) + vbar(z, mu[1])*vbar(z,mu[2]), 0))
)

a_0_eq_bar_wm_pm

Eq(chi*(ubar(z, mu[1])*ubar(z, mu[2]) + vbar(z, mu[1])*vbar(z, mu[2]))*(ubar(z, mu[1])*vbar(z, mu[1])*gamma[1] - ubar(z, mu[1])*vbar(z, mu[1])*lamda[1] + ubar(z, mu[2])*vbar(z, mu[2])*gamma[1] + ubar(z, mu[2])*vbar(z, mu[2])*lamda[1])/(8*(-1)**m*gamma[1]), 4*(-1)**m*C*b[2]/chi + 4*(-1)**m*(b[0] - b[2]*lamda[1]**2)/(C*chi) + chi*ubar(z, mu[1])**2*vbar(z, mu[1])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])**2/4 + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[1])*vbar(z, mu[1]) + (-2/chi + lamda[1]/((-1)**m*C))*ubar(z, mu[2])*vbar(z, mu[2]))

In [200]:
drhobar = Eq(
    (duvbar1.lhs**2)
    .subs([
        (ubar(z, mu[1])*vbar(z, mu[1]), gammabar[1] - rhobar(z)),
        (ubar(z, mu[2])*vbar(z, mu[2]), gammabar[2] - rhobar(z))
    ])
    .doit(),
    (
        (duvbar1.rhs**2)
        .subs((-1)**(2*m),1)
        - (
            (a_0_eq_bar_wm_pm.lhs**2 - a_0_eq_bar_wm_pm.rhs**2)
        ).subs((-1)**(2*m),1)
    )
    .subs([
        (ubar(z, mu[1])*vbar(z, mu[1]), gammabar[1] - rhobar(z)),
        (ubar(z, mu[2])*vbar(z, mu[2]), gammabar[2] - rhobar(z))
    ])
    .expand().collect(rhobar(z), factor)
    .subs(gams2_to_1_subs)
    .subs([
        ((-1)**(2*m),1),
        ((-1)**(3*m), (-1)**(m)),
        gammabar_1_to_gamma_1.args,
        C_d5_chi.args
    ])
    .subs([
        (ubar(z, mu[1])*vbar(z, mu[1]), gammabar[1] - rhobar(z)),
        (ubar(z, mu[2])*vbar(z, mu[2]), gammabar[2] - rhobar(z))
    ])
    .subs([gammabar_1_to_gamma_1.args])
    .subs(gams2_to_1_subs)
    .expand().collect(rhobar(z), factor)
    .subs([
        ((-1)**(2*m),1),
        ((-1)**(3*m), (-1)**(m)),
        gammabar_1_to_gamma_1.args,
        gam_prod_C_sqrd.args
    ])
    .collect(rhobar(z), factor)
    .subs([gam1_sqrd_chi.args])
    .collect(rhobar(z), factor)
)

drhobar

Eq(Derivative(rhobar(z), z)**2, chi**2*(C**2*d[5] - 4*C**2*gamma[1] - 4*gamma[1]**3 + 4*gamma[1]*lamda[1]**2)*rhobar(z)**3*d[5]/(64*C**4) + chi**2*(16*C**2*chi**3*b[2]*d[5]**3 - 512*C**2*chi**2*d[5]**2 + 4096*C**2*chi*b[0]*d[5] - 4096*C**2*chi*b[2]*d[5]*lamda[1]**2 + 1024*C**2*d[5]**2 - 16384*C**2*d[5]*lamda[1] + 131072*C**2*gamma[1]*lamda[1] + 131072*C**2*lamda[1]**2 - chi**4*d[5]**4 + 256*chi**2*d[5]**2*lamda[1]**2 + 131072*gamma[1]**3*lamda[1] - 131072*gamma[1]*lamda[1]**3)*rhobar(z)**2*d[5]**2/(4194304*C**6) + chi**2*(C**2*chi**3*b[2]*d[5]**4 - 8*C**2*chi**3*b[2]*d[5]**3*lamda[1] - 32*C**2*chi**2*d[5]**3 + 256*C**2*chi**2*d[5]**2*lamda[1] + 256*C**2*chi*b[0]*d[5]**2 - 2048*C**2*chi*b[0]*d[5]*lamda[1] - 256*C**2*chi*b[2]*d[5]**2*lamda[1]**2 + 2048*C**2*chi*b[2]*d[5]*lamda[1]**3 + 8192*C**2*d[5]*lamda[1]**2 - 32768*C**2*gamma[1]*lamda[1]**2 - 65536*C**2*lamda[1]**3 + chi**4*d[5]**4*lamda[1] - 256*chi**2*d[5]**2*lamda[1]**3 - 32768*gamma[1]**3*lamda[1]**2 + 32768*gamma[1]*lamda[1]**4)

In [204]:
drhobar = Eq(
    drhobar.lhs,
    drhobar.rhs
    .subs([
        C_d5_chi.args,
        ((-1)**(2*m),1),
        ((-1)**(3*m), (-1)**(m)),
        (gam1_sqrd_chi.lhs*gamma[1], gam1_sqrd_chi.rhs*gamma[1])
    ])
    .collect(rhobar(z), factor)
)
drhobar

Eq(Derivative(rhobar(z), z)**2, 4*rhobar(z)**3 + (chi**3*b[2]*d[5]**3 - 48*chi**2*d[5]**2 + 256*chi*b[0]*d[5] - 256*chi*b[2]*d[5]*lamda[1]**2 + 64*d[5]**2 - 1024*d[5]*lamda[1] + 12288*lamda[1]**2)*rhobar(z)**2/(4*chi**2*d[5]**2) + 2*(chi**3*b[2]*d[5]**4 - 8*chi**3*b[2]*d[5]**3*lamda[1] - 32*chi**2*d[5]**3 + 512*chi**2*d[5]**2*lamda[1] + 256*chi*b[0]*d[5]**2 - 2048*chi*b[0]*d[5]*lamda[1] - 256*chi*b[2]*d[5]**2*lamda[1]**2 + 2048*chi*b[2]*d[5]*lamda[1]**3 + 8192*d[5]*lamda[1]**2 - 131072*lamda[1]**3)*rhobar(z)/(chi**4*d[5]**3) + (chi**6*b[2]**2*d[5]**6 - 64*chi**5*b[2]*d[5]**5 + 512*chi**4*b[0]*b[2]*d[5]**4 - 512*chi**4*b[2]**2*d[5]**4*lamda[1]**2 + 1024*chi**4*d[5]**4 - 16384*chi**3*b[0]*d[5]**3 + 32768*chi**3*b[2]*d[5]**3*lamda[1]**2 + 65536*chi**2*b[0]**2*d[5]**2 - 131072*chi**2*b[0]*b[2]*d[5]**2*lamda[1]**2 + 65536*chi**2*b[2]**2*d[5]**2*lamda[1]**4 - 786432*chi**2*d[5]**2*lamda[1]**2 + 4194304*chi*b[0]*d[5]*lamda[1]**2 - 4194304*chi*b[2]*d[5]*lamda[1]**4 + 134217728*lamda[1]**4)/(16

In [203]:
gam1_sqrd_chi

Eq(gamma[1]**2, -chi**2*d[5]**2/256 + lamda[1]**2)

In [198]:
gamma_j_prod_b_poly
C_gam_prod
C_gam_prod_sqrd
C_gam_prod_sign
chi_d5_C
C_d5_chi
b_C_one_over_chi
d5_C_b_lam
b_long_C
gam_prod_C_sqrd
gam1_sqrd_chi



Eq(Product(gamma[j] - lamda[1], (j, 1, 2)), (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(Product(sqrt(gamma[j] - lamda[1]), (j, 1, 2)), C)

Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m*C)

Eq(chi, 16*(-1)**m*C/d[5])

...

Eq(b[1]/4 + b[2]*lamda[1]/2 - lamda[1]/(2*(-1)**m*C), 1/chi)

Eq(d[5], 4*(-1)**m*C*(b[1] + 2*b[2]*lamda[1]) - 8*lamda[1])

Eq(2*b[0]*lamda[1] + b[1]*gamma[1]**2 + b[1]*lamda[1]**2 + 2*b[2]*gamma[1]**2*lamda[1], -4*C**2/chi + 2*C*lamda[1]/(-1)**m)

Eq((gamma[1] - lamda[1])*(gamma[1] + lamda[1]), -C**2)

Eq(gamma[1]**2, -chi**2*d[5]**2/256 + lamda[1]**2)